# GLN (Graph Logic Network) 环境配置教程

## 概述

GLN（Graph Logic Network）是一种基于**条件图逻辑网络**的逆合成预测方法，发表于 NeurIPS 2019。

> **论文**: *Retrosynthesis Prediction with Conditional Graph Logic Network* (Dai et al., 2019)  
> **核心思想**: 将逆合成预测分解为三个层次化的子问题：反应中心预测 → 模板选择 → 反应物验证，每个子问题都用图神经网络 (GNN) 来编码分子/模板结构，并通过注意力机制进行评分。

### 本教程包含三份 notebook

| 序号 | Notebook | 内容 |
|------|----------|------|
| 1 | **环境配置**（本文件） | 安装依赖、验证环境 |
| 2 | 数据处理 | 最小化的 GLN 数据处理流程 |
| 3 | 模型展示 | GLN 推理原理与模型计算架构 |

---

## 1. 核心依赖一览

GLN 原始代码依赖较老的版本（RDKit 2019_03_3, PyTorch 1.2-1.3）。本教程适配到当前主流版本：

| 依赖库 | 原始版本 | 教程版本（推荐） | 说明 |
|--------|----------|-----------------|------|
| Python | 3.6-3.7 | **3.10+** | 现代化 Python |
| PyTorch | 1.2-1.3 | **2.0+** | 新版 PyTorch |
| RDKit | 2019_03_3 | **2023.09+** | 化学信息学工具包 |
| torch-scatter | 旧版 | **2.1+** | 图操作scatter算子 |
| torch-geometric | 旧版 | **2.4+** | 图神经网络框架 |
| numpy | 旧版 | **1.24+** | 数值计算 |
| scipy | 旧版 | **1.10+** | 科学计算 |

> **注意**: GLN 原始代码包含 C++ 扩展（extlib, mg_clib）用于性能加速。在教程中我们使用纯 Python 实现来展示核心逻辑，不需要编译 C++ 代码。

## 2. 创建虚拟环境并安装依赖

以下代码会在项目的 `envs/` 目录下创建一个独立的 Python 虚拟环境，避免与系统环境冲突。

In [1]:
import os
from pathlib import Path

# ====== 定位项目根目录 ======
def find_project_root(start=None):
    """向上查找项目根目录（包含 teaching_demos/ 的目录）"""
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / 'teaching_demos').exists():
            return candidate
    raise FileNotFoundError('无法定位项目根目录')

PROJECT_ROOT = find_project_root()
ENV_DIR = PROJECT_ROOT / 'envs' / 'gln_tutorial_envs'
SOURCE_REPO = PROJECT_ROOT / 'source_repos' / 'gln'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'虚拟环境目录: {ENV_DIR}')
print(f'GLN 源码目录: {SOURCE_REPO}')

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
虚拟环境目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs
GLN 源码目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/gln


In [ ]:
%%bash -s "$ENV_DIR"

ENV_DIR=$1

# 让 bash 能识别 conda 命令
source "$(conda info --base)/etc/profile.d/conda.sh"

# 如果虚拟环境不存在则创建
if [ ! -d "$ENV_DIR" ]; then
    echo "==> 正在创建 Conda 环境 $ENV_DIR ..."
    conda create -y -p "$ENV_DIR" python=3.11
    echo "==> Conda 环境创建完成"
else
    echo "==> Conda 环境已存在: $ENV_DIR"
fi

# 激活环境
conda activate "$ENV_DIR"

echo "==> 安装核心依赖 ..."
conda install -y -c conda-forge \
    numpy \
    scipy \
    pandas \
    tqdm \
    rdkit \
    matplotlib \
    ipykernel

echo "==> 安装 PyTorch 相关依赖 ..."
pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cpu.html
pip install torch_geometric
# pip install -q \
#     torch-geometric

echo "==> 所有依赖安装完成"

==> 正在创建 Conda 环境 /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs ...
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs

  added / updated specs:
    - python=3.11


The following NEW packages will be INSTALLED:

  _libgcc_mutex      anaconda/pkgs/main/linux-64::_libgcc_mutex-0.1-main 
  _openmp_mutex      anaconda/pkgs/main/linux-64::_openmp_mutex-5.1-1_gnu 
  bzip2              anaconda/pkgs/main/linux-64::bzip2-1.0.8-h5eee18b_6 
  ca-certificates    anaconda/pkgs/main/linux-64::ca-certificates-2025.12.2-h06a4308_0 
  expat              anaconda/pkgs/main/linux-64::expat-2.7.4-h7354ed3_0 
  ld_impl_linux-64   anaconda/pkgs/main/linux-64::ld_impl_linux-64-2.44-h153f514_2 
  libexpat           anaconda/pkgs/main/linux-64::libexpat-2.7.4-h7354ed3_0 
  lib



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs

  added / updated specs:
    - ipykernel
    - matplotlib
    - numpy
    - pandas
    - rdkit
    - scipy
    - tqdm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    contourpy-1.3.3            |  py311h724c32c_4         313 KB  conda-forge
    debugpy-1.8.20             |  py311hc665b79_0         2.6 MB  conda-forge
    fonttools-4.62.0           |  py311h3778330_0         2.9 MB  conda-forge
    greenlet-3.3.2             |  py311hc665b79_0         253 KB  conda-forge
    ipython-9.10.0             |     pyh53cf698_0         632 KB  conda-forge
    kiwisolver-1.5.0           |  py311h724c32c_0          76 KB  conda-forge
    libboost-python-1.86.0     |  py311h1d5f577_5         117 KB  conda-forge
    matplotlib-3.10.8          |  py311h

ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib


Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.3 MB ? eta -:--:--━━━━━━━━━━━━━━━━━━ 0.3/1.3 MB ? eta -:--:--━━━━╸━━━━━━━ 1.0/1.3 MB 5.0 MB/s eta 0:00:01━━━━╸━━━━━━━ 1.0/1.3 MB 5.0 MB/s eta 0:00:01━━━━╸━━━━━━━ 1.0/1.3 MB 5.0 MB/s eta 0:00:01━━━━╸━━━━━━━ 1.0/1.3 MB 5.0 MB/s eta 0:00:01━━━━━━━━━━━━ 1.3/1.3 MB 1.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━0m━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--━━━━━━━━━━━━━━━━━━ 0.5/1.7 MB 3.3 MB/s eta 0:00:01━━━━━━━╸━━━━ 1.6/1.7 MB 4.1 MB/s eta 0:00:01━━━━━━━━━━━━ 1.7/1.7 MB 3.5 MB/s  0:00:00
  Using cached https://mirrors.tuna.tsinghua.edu.cn/pypi/web/packages/1e/db/4254e3eabe8020b458f1a747140d32277ec7a271daf1d235b70dc0b4e6e3/requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached https://mirrors.tuna.tsinghua.edu.cn/pypi/web/packages/3a/2a/7cc015f5b9f5db42b7d48157e23356022889fc354a2813c15934b7cb5c0e/attrs-25.4.0-py3-none-any.whl (67 kB)
  Using cached https://mirrors.tu

### 将虚拟环境注册为 Jupyter Kernel（可选）

如果你在 Jupyter 环境中运行，可以将新环境注册为独立的 kernel：

In [3]:
%%bash -s "$ENV_DIR"
source "$1/bin/activate"
python -m ipykernel install --user --name gln_tutorial --display-name "GLN Tutorial"
echo "==> Kernel 'GLN Tutorial' 注册完成，请在 Jupyter 中选择该 kernel"

bash: 行 1: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs/bin/activate: 没有那个文件或目录


Installed kernelspec gln_tutorial in /home/xiaoruiwang/.local/share/jupyter/kernels/gln_tutorial
==> Kernel 'GLN Tutorial' 注册完成，请在 Jupyter 中选择该 kernel


## 3. 验证环境

运行以下代码验证所有核心依赖是否安装成功。

> **提示**: 确保当前 notebook 使用的是 `gln_tutorial` kernel 或对应虚拟环境中的 Python。

In [1]:
import sys
print(f'Python 版本: {sys.version}')
print(f'Python 路径: {sys.executable}')
print()

Python 版本: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Python 路径: /home/xiaoruiwang/data/ubuntu_work_beta/other_work/GNN_AIDD/Chemical_Synthesis/envs/gln_tutorial_envs/bin/python



In [2]:
# ====== 逐一检查依赖 ======

checks = {}

# 1. PyTorch
try:
    import torch
    checks['PyTorch'] = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except ImportError as e:
    checks['PyTorch'] = f'FAIL: {e}'

# 2. RDKit
try:
    from rdkit import Chem, rdBase
    checks['RDKit'] = rdBase.rdkitVersion
except ImportError as e:
    checks['RDKit'] = f'FAIL: {e}'

# 3. NumPy
try:
    import numpy as np
    checks['NumPy'] = np.__version__
except ImportError as e:
    checks['NumPy'] = f'FAIL: {e}'

# 4. SciPy
try:
    import scipy
    checks['SciPy'] = scipy.__version__
except ImportError as e:
    checks['SciPy'] = f'FAIL: {e}'

# 5. torch-scatter
try:
    from torch_scatter import scatter_add
    checks['torch-scatter'] = 'OK'
except ImportError as e:
    checks['torch-scatter'] = f'FAIL: {e}'

# 6. torch-geometric
try:
    import torch_geometric
    checks['torch-geometric'] = torch_geometric.__version__
except ImportError as e:
    checks['torch-geometric'] = f'FAIL: {e}'

# 7. pandas
try:
    import pandas as pd
    checks['pandas'] = pd.__version__
except ImportError as e:
    checks['pandas'] = f'FAIL: {e}'

# 8. matplotlib
try:
    import matplotlib
    checks['matplotlib'] = matplotlib.__version__
except ImportError as e:
    checks['matplotlib'] = f'FAIL: {e}'

# 9. tqdm
try:
    import tqdm
    checks['tqdm'] = tqdm.__version__
except ImportError as e:
    checks['tqdm'] = f'FAIL: {e}'

print('='*50)
print('环境检查结果')
print('='*50)
all_ok = True
for name, status in checks.items():
    flag = '✓' if 'FAIL' not in str(status) else '✗'
    if 'FAIL' in str(status):
        all_ok = False
    print(f'  {flag} {name:20s} {status}')

print('='*50)
if all_ok:
    print('所有依赖检查通过！')
else:
    print('部分依赖缺失，请检查上方安装步骤。')

环境检查结果
  ✓ PyTorch              2.10.0+cpu (CUDA: False)
  ✓ RDKit                2025.09.6
  ✓ NumPy                2.4.2
  ✓ SciPy                1.17.1
  ✓ torch-scatter        OK
  ✓ torch-geometric      2.7.0
  ✓ pandas               3.0.1
  ✓ matplotlib           3.10.8
  ✓ tqdm                 4.67.3
所有依赖检查通过！


## 4. 验证 RDKit 基础功能

GLN 的核心操作依赖 RDKit 的以下功能：
- **SMILES 解析与标准化**: 将分子表示统一为 canonical 形式
- **SMARTS 子结构匹配**: 用于匹配反应中心
- **原子特征提取**: 用于构建分子图

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

# ====== 4.1 SMILES 解析与标准化 ======
# GLN 数据处理的第一步就是将所有 SMILES 标准化

def cano_smiles(smiles):
    """将 SMILES 标准化：去氢、去原子映射编号"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, smiles
        mol = Chem.RemoveHs(mol)
        if mol is None:
            return None, smiles
        [a.ClearProp('molAtomMapNumber') for a in mol.GetAtoms()]
        return mol, Chem.MolToSmiles(mol)
    except:
        return None, smiles

# 测试标准化
test_smiles = [
    'c1ccccc1',           # 苯 (Kekulé)
    'C1=CC=CC=C1',       # 苯 (另一种写法)
    '[CH3:1]O',          # 带原子映射的甲醇
]

print('SMILES 标准化示例：')
print('-' * 50)
for sm in test_smiles:
    mol, cano = cano_smiles(sm)
    print(f'  原始: {sm:20s} → 标准化: {cano}')

SMILES 标准化示例：
--------------------------------------------------
  原始: c1ccccc1             → 标准化: c1ccccc1
  原始: C1=CC=CC=C1          → 标准化: c1ccccc1
  原始: [CH3:1]O             → 标准化: CO


In [4]:
# ====== 4.2 SMARTS 子结构匹配 ======
# GLN 用 SMARTS 模式来表示反应中心，并在产物分子上做子结构匹配

def cano_smarts(smarts):
    """标准化 SMARTS 模式"""
    mol = Chem.MolFromSmarts(smarts)
    if mol is None:
        return None, smarts
    [a.ClearProp('molAtomMapNumber') for a in mol.GetAtoms()]
    cano = Chem.MolToSmarts(mol)
    if '[[se]]' in cano:  # RDKit 已知bug
        cano = smarts
    return mol, cano

# 示例：羰基 SMARTS 匹配
smarts_pattern = '[C:1](=[O:2])'
product_smiles = 'CC(=O)c1ccccc1'  # 苯乙酮

center_mol = Chem.MolFromSmarts(smarts_pattern)
prod_mol = Chem.MolFromSmiles(product_smiles)

has_match = prod_mol.HasSubstructMatch(center_mol)
matches = prod_mol.GetSubstructMatches(center_mol)

print(f'SMARTS 模式: {smarts_pattern}')
print(f'产物 SMILES: {product_smiles}')
print(f'是否匹配: {has_match}')
print(f'匹配位置: {matches}')

SMARTS 模式: [C:1](=[O:2])
产物 SMILES: CC(=O)c1ccccc1
是否匹配: True
匹配位置: ((1, 2),)


In [5]:
# ====== 4.3 原子/键特征提取 ======
# GLN 将分子表示为图结构，节点 = 原子，边 = 化学键
# 每个原子/键都编码为一组离散特征

import rdkit
from rdkit import Chem

mol = Chem.MolFromSmiles('CC(=O)c1ccccc1')  # 苯乙酮

print('='*60)
print('原子特征（GLN 使用的特征）')
print('='*60)
print(f'{"原子":>4} {"符号":>4} {"原子序数":>8} {"度数":>4} {"氢计数":>6} {"隐式化合价":>10} {"芳香性":>6}')
print('-'*60)
for atom in mol.GetAtoms():
    print(f'{atom.GetIdx():4d} {atom.GetSymbol():>4} {atom.GetAtomicNum():8d} '
          f'{atom.GetDegree():4d} {atom.GetTotalNumHs():6d} '
          f'{atom.GetImplicitValence():10d} {str(atom.GetIsAromatic()):>6}')

print()
print('='*60)
print('化学键特征')
print('='*60)
print(f'{"键":>4} {"起点":>4} {"终点":>4} {"键类型":>10} {"共轭":>6} {"环内":>6}')
print('-'*60)
for bond in mol.GetBonds():
    ring_info = mol.GetRingInfo()
    in_ring = ring_info.NumBondRings(bond.GetIdx()) > 0
    print(f'{bond.GetIdx():4d} {bond.GetBeginAtomIdx():4d} {bond.GetEndAtomIdx():4d} '
          f'{str(bond.GetBondType()):>10} {str(bond.GetIsConjugated()):>6} {str(in_ring):>6}')

原子特征（GLN 使用的特征）
  原子   符号     原子序数   度数    氢计数      隐式化合价    芳香性
------------------------------------------------------------
   0    C        6    1      3          3  False
   1    C        6    3      0          0  False
   2    O        8    1      0          0  False
   3    C        6    3      0          0   True
   4    C        6    2      1          1   True
   5    C        6    2      1          1   True
   6    C        6    2      1          1   True
   7    C        6    2      1          1   True
   8    C        6    2      1          1   True

化学键特征
   键   起点   终点        键类型     共轭     环内
------------------------------------------------------------
   0    0    1     SINGLE  False  False
   1    1    2     DOUBLE   True  False
   2    1    3     SINGLE   True  False
   3    3    4   AROMATIC   True   True
   4    4    5   AROMATIC   True   True
   5    5    6   AROMATIC   True   True
   6    6    7   AROMATIC   True   True
   7    7    8   AROMATIC   True   True
   8 

[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:58:45] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead


## 5. 验证 PyTorch 图操作

GLN 使用 `torch-scatter` 和 `torch-geometric` 来实现图神经网络中的消息传递操作。

In [6]:
import torch
from torch_scatter import scatter_add, scatter_mean

# ====== scatter 操作示例 ======
# scatter 是 GNN 消息传递的核心：将邻居节点的信息聚合到目标节点

# 假设有 5 个节点，分属 2 个图
node_feats = torch.tensor([
    [1.0, 0.0],  # 图0-节点0
    [0.0, 1.0],  # 图0-节点1
    [1.0, 1.0],  # 图0-节点2
    [2.0, 0.0],  # 图1-节点0
    [0.0, 2.0],  # 图1-节点1
])

# 每个节点所属的图索引
graph_idx = torch.tensor([0, 0, 0, 1, 1])

# scatter_add: 对同一个图中的节点特征求和 → 得到图级别的表示
graph_embed_sum = scatter_add(node_feats, graph_idx, dim=0, dim_size=2)
graph_embed_mean = scatter_mean(node_feats, graph_idx, dim=0, dim_size=2)

print('scatter_add（图级别求和）:')
print(f'  图0: {graph_embed_sum[0].tolist()}  (= [1,0] + [0,1] + [1,1] = [2,2])')
print(f'  图1: {graph_embed_sum[1].tolist()}  (= [2,0] + [0,2] = [2,2])')
print()
print('scatter_mean（图级别平均）:')
print(f'  图0: {graph_embed_mean[0].tolist()}')
print(f'  图1: {graph_embed_mean[1].tolist()}')

scatter_add（图级别求和）:
  图0: [2.0, 2.0]  (= [1,0] + [0,1] + [1,1] = [2,2])
  图1: [2.0, 2.0]  (= [2,0] + [0,2] = [2,2])

scatter_mean（图级别平均）:
  图0: [0.6666666865348816, 0.6666666865348816]
  图1: [1.0, 1.0]


In [7]:
from torch_geometric.nn.conv import MessagePassing

# ====== 简单的消息传递示例 ======
# GLN 的 EmbedMeanField 就是一种简单的消息传递 GNN

class SimpleMeanFieldLayer(MessagePassing):
    """最简消息传递层：聚合邻居信息后线性变换"""
    def __init__(self, dim):
        super().__init__(aggr='add')  # 使用加法聚合
        self.linear = torch.nn.Linear(dim, dim)
    
    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)
    
    def update(self, aggr_out):
        return self.linear(aggr_out)

# 构建一个简单的分子图 (丙烷 C-C-C)
#   0 -- 1 -- 2
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]])  # 无向边
x = torch.randn(3, 4)  # 3个节点，4维特征

layer = SimpleMeanFieldLayer(4)
out = layer(x, edge_index)

print(f'输入特征形状: {x.shape}')
print(f'输出特征形状: {out.shape}')
print(f'消息传递层参数量: {sum(p.numel() for p in layer.parameters())}')
print('\n✓ torch-geometric 消息传递机制正常工作！')

输入特征形状: torch.Size([3, 4])
输出特征形状: torch.Size([3, 4])
消息传递层参数量: 20

✓ torch-geometric 消息传递机制正常工作！


## 6. GLN 源代码结构概览

为了方便后续教程的理解，这里展示 GLN 的源代码目录结构及各模块功能。

In [8]:
# 打印 GLN 代码结构
structure = """
GLN 源代码结构
══════════════════════════════════════════════════════════════
gln/
├── common/                   # 公共工具
│   ├── cmd_args.py           # 命令行参数定义
│   ├── consts.py             # 全局常量（设备、数据类型）
│   ├── evaluate.py           # 评估指标（精确匹配、F1）
│   ├── mol_utils.py          # 分子工具（SMILES/SMARTS标准化）
│   ├── reactor.py            # 模板应用器（rdChiral封装）
│   └── torch_util.py         # PyTorch工具（MLP等）
│
├── data_process/             # 数据处理流水线
│   ├── get_canonical_smiles.py    # Step 0: SMILES标准化
│   ├── build_raw_template.py      # Step 1: 提取反应模板
│   ├── filter_template.py         # Step 2: 过滤低频模板
│   ├── get_canonical_smarts.py    # Step 3: SMARTS标准化
│   ├── find_centers.py            # Step 4: 查找反应中心
│   ├── build_all_reactions.py     # Step 5: 构建所有可能的反应
│   ├── dump_graphs.py             # Step 6: 导出图特征
│   └── data_info.py              # 数据管理器（DataInfo类）
│
├── graph_logic/              # 核心模型
│   ├── __init__.py           # GNN工厂函数
│   ├── graph_feat.py         # 模板/反应物特征编码器
│   ├── logic_net.py          # GraphPath主模型
│   └── soft_logic.py         # 三个概率计算模块
│
├── mods/                     # 底层模块
│   ├── mol_gnn/              # 分子图神经网络
│   │   ├── gnn_family/       # GNN变体（MeanField/GGNN/MPNN等）
│   │   ├── mol_utils.py      # 分子图数据结构
│   │   └── torch_util.py     # MLP、激活函数
│   └── torchext/             # C++扩展（加速用）
│
├── training/                 # 训练脚本
│   ├── main.py               # 训练主循环
│   └── data_gen.py           # 数据生成器
│
└── test/                     # 测试/推理
    ├── model_inference.py    # RetroGLN推理类
    └── main_test.py          # 评估脚本
"""
print(structure)


GLN 源代码结构
══════════════════════════════════════════════════════════════
gln/
├── common/                   # 公共工具
│   ├── cmd_args.py           # 命令行参数定义
│   ├── consts.py             # 全局常量（设备、数据类型）
│   ├── evaluate.py           # 评估指标（精确匹配、F1）
│   ├── mol_utils.py          # 分子工具（SMILES/SMARTS标准化）
│   ├── reactor.py            # 模板应用器（rdChiral封装）
│   └── torch_util.py         # PyTorch工具（MLP等）
│
├── data_process/             # 数据处理流水线
│   ├── get_canonical_smiles.py    # Step 0: SMILES标准化
│   ├── build_raw_template.py      # Step 1: 提取反应模板
│   ├── filter_template.py         # Step 2: 过滤低频模板
│   ├── get_canonical_smarts.py    # Step 3: SMARTS标准化
│   ├── find_centers.py            # Step 4: 查找反应中心
│   ├── build_all_reactions.py     # Step 5: 构建所有可能的反应
│   ├── dump_graphs.py             # Step 6: 导出图特征
│   └── data_info.py              # 数据管理器（DataInfo类）
│
├── graph_logic/              # 核心模型
│   ├── __init__.py           # GNN工厂函数
│   ├── graph_feat.py         # 模板/反应物特征编码器
│   ├── l

## 7. 快速理解 GLN 的工作流程

GLN 的整体流程可以总结为以下管线：

```
┌────────────────────────────────────────────────────────────────┐
│                        训练阶段                                │
│                                                                │
│  原始反应数据                                                   │
│    ↓                                                           │
│  SMILES 标准化 → 模板提取 → SMARTS 标准化                       │
│    ↓                                                           │
│  反应中心匹配 → 构建正负样本 → 图特征导出                        │
│    ↓                                                           │
│  训练 GraphPath 模型（三阶段联合训练）                           │
└────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────┐
│                        推理阶段                                │
│                                                                │
│  输入：产物 SMILES                                              │
│    ↓                                                           │
│  (1) 反应中心预测：GNN编码产物 → 对候选中心评分                  │
│    ↓                                                           │
│  (2) 模板选择：GNN编码模板 → 与产物做注意力匹配                  │
│    ↓                                                           │
│  (3) 反应物生成：应用模板 → GNN验证 → 输出Top-K预测             │
└────────────────────────────────────────────────────────────────┘
```

这里的三个阶段对应论文中的三个条件概率：

$$P(\text{reactants} | \text{product}) = \sum_{c,t} P(c|p) \cdot P(t|c,p) \cdot P(r|t,p)$$

其中：
- $P(c|p)$：给定产物 $p$，反应中心 $c$ 的概率
- $P(t|c,p)$：给定产物和中心，模板 $t$ 的概率
- $P(r|t,p)$：给定产物和模板，反应物 $r$ 的概率

## 8. 总结

本 notebook 完成了以下工作：

1. ✅ 安装了 GLN 教程所需的所有 Python 依赖
2. ✅ 验证了 RDKit、PyTorch、torch-geometric 等核心库
3. ✅ 演示了 GLN 用到的 RDKit 核心操作（SMILES标准化、SMARTS匹配、特征提取）
4. ✅ 演示了 GLN 用到的 PyTorch图操作（scatter、MessagePassing）
5. ✅ 概览了 GLN 的代码结构和工作流程

**下一步** → 请打开 `2_数据处理.ipynb`，学习 GLN 的完整数据处理流程。